# Set up environment

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install biopython peptides pandas peptides
!apt-get install ncbi-blast+ cd-hit -y
!pip install -q tensorflow accelerate safetensors

In [ ]:
!pip install toxinpred3

In [ ]:
!git clone https://github.com/bcgsc/AMPlify.git /content/AMPlify

import os
if not os.path.exists("/content/AMPlify"):
    !git clone https://github.com/bcgsc/AMPlify.git /content/AMPlify

In [ ]:
!pip install scikit-learn==1.2.2 --force-reinstall --no-deps

In [ ]:
!pip install "numpy<2.0.0"

In [ ]:
import sklearn
print(f"Active scikit-learn version: {sklearn.__version__}")

# Phase 1

In [ ]:
import pandas as pd
import peptides
from Bio import SeqIO

class Phase1_PhysicochemicalGate:
    """
    Phase 1: Binary Pass/Fail Gate. Acts as a strict sanity check.
    """
    def __init__(self, results: pd.DataFrame):
        self.results = results

    def evaluate(self) -> pd.DataFrame:
        print("Running Phase 1: Physicochemical Sanity Gate...")

        hydrophobic_aas = set("VILFMWAY")

        def gate_and_features(seq):
            default = pd.Series({
                "phase1_pass":               False,
                "phase1_length":            0.0,
                "phase1_charge":            0.0,
                "phase1_hydrophobic_frac":  0.0,
                "phase1_hydrophobic_moment": 0.0
            })

            # HARD DISQUALIFIER: Length
            if not (6 <= len(seq) <= 100):
                return default

            pep = peptides.Peptide(seq)
            charge = pep.charge(pH=7.4)

            # HARD DISQUALIFIER: Net Charge
            if charge < 0.0:
                return default

            # HARD DISQUALIFIER: Hydrophobic Fraction
            hydro_frac = sum(1 for aa in seq if aa in hydrophobic_aas) / len(seq)
            if not (0.15 <= hydro_frac <= 0.75):
                return default

            return pd.Series({
                "phase1_pass":               True,
                "phase1_length":             float(len(seq)),
                "phase1_charge":             round(charge, 2),
                "phase1_hydrophobic_frac":   round(hydro_frac, 3),
                "phase1_hydrophobic_moment": round(pep.hydrophobic_moment(), 4)
            })

        feature_cols = [
            "phase1_pass",
            "phase1_length",
            "phase1_charge",
            "phase1_hydrophobic_frac",
            "phase1_hydrophobic_moment"
        ]

        features = self.results["sequence"].apply(gate_and_features)

        for col in feature_cols:
            self.results[col] = features[col]

        n_passed = self.results["phase1_pass"].sum()
        print(f"  ✓ Phase 1 Complete: {n_passed}/{len(self.results)} sequences passed.")
        return self.results

# Phase 2

### GRAMPA

In [ ]:
import os
import pandas as pd
import subprocess

def setup_grampa_blast_database():
    """
    Downloads the raw GRAMPA CSV file, extracts unique active AMP sequences,
    and builds an optimized local BLAST reference database.
    """
    db_fasta = "amp_reference_db.fasta"
    db_name = "amp_db"

    grampa_url = "https://raw.githubusercontent.com/zswitten/Antimicrobial-Peptides/master/data/grampa.csv"

    print("Fetching and building BLAST database from GRAMPA dataset...")

    try:
        df_grampa = pd.read_csv(grampa_url)

        unique_sequences = df_grampa['sequence'].dropna().unique()
        print(f"  ✓ Found {len(unique_sequences)} unique, verified reference AMPs.")

        with open(db_fasta, "w") as f:
            for i, seq in enumerate(unique_sequences):
                f.write(f">grampa_{i}\n{seq.upper()}\n")
        print("  ✓ Converted entries to 'amp_reference_db.fasta'.")

        cmd = ["makeblastdb", "-in", db_fasta, "-dbtype", "prot", "-out", db_name]
        subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT)
        print(f"  ✓ Local BLAST database '{db_name}' compiled and ready.")

        return db_name

    except Exception as e:
        raise RuntimeError(f"Failed to process GRAMPA database: {str(e)}")

AMP_DB_PATH = setup_grampa_blast_database()

In [ ]:
import os
import math
import subprocess
from collections import Counter
import pandas as pd
from Bio.Blast import NCBIXML


class Phase2_NoveltyScorer:
    """
    Phase 2: Novelty Scorer (0–100).
    Combines BLAST similarity and Shannon entropy via geometric mean to find unique sequences.
    """

    def __init__(self, results: pd.DataFrame, db_path: str = "amp_db"):
        self.results = results
        self.db_path = db_path


    def evaluate(
        self,
        cd_hit_identity: float = 0.95,
        min_entropy: float = 2.0,
        max_entropy: float = 5.0,
    ) -> pd.DataFrame:

        mask = self.results["phase1_pass"] == True
        candidates = self.results.loc[mask, "sequence"].copy()

        if candidates.empty:
            print("Phase 2: No sequences passed Phase 1. Skipping.")
            return self.results

        print("Running Phase 2: Novelty Scorer...")

        blast_scores  = self._run_blast(candidates)
        entropy_scores = candidates.apply(
            lambda seq: self._entropy_score(seq, min_entropy, max_entropy)
        )

        novelty = (blast_scores * entropy_scores).pow(0.5).round(2)

        self.results.loc[mask, "phase2_blast_identity"]  = blast_scores.apply(
            lambda s: round(100 - s, 2)
        )
        self.results.loc[mask, "phase2_entropy"]         = candidates.apply(
            lambda seq: round(self._shannon_entropy(seq), 4)
        )
        self.results.loc[mask, "phase2_score"]           = novelty

        print(f"    Phase 2 Complete: scored {mask.sum()} sequences.")
        print(f"    Mean novelty score : {novelty.mean():.1f}")
        print(f"    Sequences > 80     : {(novelty > 80).sum()}")
        print(f"    Sequences < 20     : {(novelty < 20).sum()}\n")

        return self.results


    def _run_blast(self, candidates: pd.Series) -> pd.Series:
        query_fasta = "_phase2_query.fasta"
        out_xml     = "_phase2_blast.xml"

        with open(query_fasta, "w") as f:
            for idx, seq in candidates.items():
                f.write(f">seq_{idx}\n{seq}\n")

        cmd = [
            "blastp",
            "-query",  query_fasta,
            "-db",     self.db_path,
            "-evalue", "10",
            "-outfmt", "5",
            "-out",    out_xml,
            "-task",   "blastp-short",
        ]
        subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

        identity_map = {idx: 0.0 for idx in candidates.index}

        if os.path.exists(out_xml):
            with open(out_xml) as h:
                for record in NCBIXML.parse(h):
                    idx = int(record.query.split("_")[1])
                    if record.alignments:
                        hsp = record.alignments[0].hsps[0]
                        identity_map[idx] = (hsp.identities / hsp.align_length) * 100.0

        for fp in [query_fasta, out_xml]:
            if os.path.exists(fp):
                os.remove(fp)

        def identity_to_score(identity_pct: float) -> float:
            return round(100.0 * (1.0 - identity_pct / 100.0) ** 2, 2)

        return pd.Series(
            {idx: identity_to_score(identity_map[idx]) for idx in candidates.index}
        )


    @staticmethod
    def _shannon_entropy(seq: str, n: int = 3) -> float:
        if len(seq) < n:
            return 0.0
        ngrams = [seq[i:i+n] for i in range(len(seq) - n + 1)]
        counts = Counter(ngrams)
        total  = len(ngrams)
        return -sum((c / total) * math.log2(c / total) for c in counts.values())

    def _entropy_score(self, seq: str, min_entropy: float, max_entropy: float) -> float:
        h = self._shannon_entropy(seq)
        if h <= min_entropy:
            return max(0.0, round((h / min_entropy) * 10.0, 2))
        score = (h - min_entropy) / (max_entropy - min_entropy) * 100.0
        return round(min(score, 100.0), 2)

# Phase 3

### AMPLIFY

In [ ]:
import os
import sys
import glob
import tempfile
import subprocess
import numpy as np
import pandas as pd


class Phase3_AMPlifyScorer:
    """
    Scores sequences using the AMPlify Attention/CNN neural network.
    """

    def __init__(self, results: pd.DataFrame, amplify_path: str = "/content/AMPlify/src/AMPlify.py"):
        self.results      = results
        self.amplify_path = amplify_path

        if "sequence" not in self.results.columns:
            raise ValueError("Master DataFrame must contain a 'sequence' column.")

        self._apply_keras_compatibility_patches()


    def evaluate(self) -> pd.DataFrame:
        mask = self.results["phase1_pass"] == True
        candidates = self.results.loc[mask, "sequence"]

        if candidates.empty:
            print("Phase 3: No sequences passed Phase 1. Skipping.")
            return self.results

        print("Running Phase 3: AMPlify Deep Neural Network Scorer...")

        with tempfile.TemporaryDirectory() as tmpdir:
            fasta_path = os.path.join(tmpdir, "input_seqs.fasta")
            out_dir    = os.path.join(tmpdir, "amplify_out")
            os.makedirs(out_dir, exist_ok=True)

            self._write_temp_fasta(candidates, fasta_path)

            cmd = [
                sys.executable, self.amplify_path,
                "-s",  fasta_path,
                "-od", out_dir,
                "-of", "tsv",
            ]

            try:
                subprocess.run(cmd, capture_output=True, text=True, check=True)
            except subprocess.CalledProcessError as e:
                print(f"AMPlify Execution Failed:\n{e.stderr}")
                raise RuntimeError("AMPlify model failed to process the sequences.")

            tsv_files = glob.glob(os.path.join(out_dir, "*.tsv"))
            if not tsv_files:
                raise FileNotFoundError("AMPlify ran but produced no output TSV.")

            df_out = pd.read_csv(tsv_files[0], sep="\t")

        scored = 0
        for _, row in df_out.iterrows():
            original_idx = int(str(row["Sequence_ID"]).split("_")[1])
            if original_idx not in self.results.index:
                continue
            raw_score  = float(row["AMPlify_log_scaled_score"])
            phase3_score = self._amplify_to_score(raw_score)
            self.results.at[original_idx, "phase3_score"] = phase3_score
            scored += 1

        print(f"    Phase 3 Complete: {scored}/{mask.sum()} sequences scored by AMPlify.")
        print(f"    Mean phase3_score : {self.results.loc[mask, 'phase3_score'].mean():.1f}\n")

        return self.results


    @staticmethod
    def _amplify_to_score(log_scaled_score: float) -> float:
        if pd.isna(log_scaled_score):
            return 0.0
        prob = 1.0 / (1.0 + np.exp(-0.1 * (log_scaled_score - 15)))
        return round(prob * 100.0, 2)


    @staticmethod
    def _write_temp_fasta(candidates: pd.Series, path: str):
        with open(path, "w") as fh:
            for idx, seq in candidates.items():
                fh.write(f">seq_{idx}\n{seq}\n")

    def _apply_keras_compatibility_patches(self):
        base_dir   = os.path.dirname(os.path.dirname(self.amplify_path))
        layers_path = os.path.join(base_dir, "src", "layers.py")

        if not os.path.exists(layers_path):
            print(f"Could not find layers.py at {layers_path}. Skipping patch.")
            return

        with open(layers_path, "r") as f:
            src = f.read()

        if "K.batch_dot(" not in src and "K.int_shape(" not in src:
            return

        print("Applying TensorFlow / Keras compatibility patches to AMPlify layers...")

        helper_functions = """
def _batch_dot(x, y, axes=None):
    if axes is not None:
        if isinstance(axes, int):
            axes = [axes, axes]
        if axes[1] == 2:
            y = tf.transpose(y, perm=[0, 2, 1])
        elif axes[0] == 1:
            x = tf.transpose(x, perm=[0, 2, 1])
    return tf.matmul(x, y)

def _int_shape(x):
    return tuple(x.shape.as_list())
"""

        if "import tensorflow as tf" not in src:
            src = "import tensorflow as tf\n" + helper_functions + src
        else:
            src = src.replace(
                "import tensorflow as tf",
                "import tensorflow as tf\n" + helper_functions,
                1,
            )

        replacements = {
            "K.batch_dot("         : "_batch_dot(",
            "K.int_shape("         : "_int_shape(",
            "K.permute_dimensions(": "tf.transpose(",
            "K.tile("              : "tf.tile(",
            "K.maximum("           : "tf.maximum(",
            "K.arange("            : "tf.range(",
            "K.dot("               : "tf.linalg.matmul(",
            "K.shape("             : "tf.shape(",
            "K.reshape("           : "tf.reshape(",
            "K.transpose("         : "tf.transpose(",
            "K.expand_dims("       : "tf.expand_dims(",
            "K.squeeze("           : "tf.squeeze(",
            "K.concatenate("       : "tf.concat(",
            "K.softmax("           : "tf.nn.softmax(",
            "K.tanh("              : "tf.math.tanh(",
            "K.relu("              : "tf.nn.relu(",
            "K.cast("              : "tf.cast(",
            "K.sum("               : "tf.reduce_sum(",
            "K.mean("              : "tf.reduce_mean(",
            "K.max("               : "tf.reduce_max(",
            "K.min("               : "tf.reduce_min(",
            "K.zeros_like("        : "tf.zeros_like(",
            "K.ones_like("         : "tf.ones_like(",
            "K.zeros("             : "tf.zeros(",
            "K.ones("              : "tf.ones(",
            "K.abs("               : "tf.abs(",
            "K.exp("               : "tf.exp(",
            "K.log("               : "tf.math.log(",
            "K.sqrt("              : "tf.sqrt(",
            "K.clip("              : "tf.clip_by_value(",
            "K.epsilon("           : "tf.keras.backend.epsilon(",
            "K.floatx("            : "tf.keras.backend.floatx(",
            "K.variable("          : "tf.Variable(",
            "K.constant("          : "tf.constant(",
        }

        for old, new in replacements.items():
            src = src.replace(old, new)

        src = src.replace(
            "tf.expand_dims(self.context)",
            "tf.expand_dims(self.context, axis=-1)",
        )

        with open(layers_path, "w") as f:
            f.write(src)

        print("  Patches applied successfully.")

# Phase 4

### ToxinPred 3.0

In [ ]:
import os
import tempfile
import numpy as np
import pandas as pd

import toxinpred3.python_scripts.toxinpred3 as tp


class Phase4_ToxicityScorer:
    """
    Converts ML toxicity values into a 0–100 safety score where higher is safer.
    """

    def __init__(self, results: pd.DataFrame):
        self.results = results

        if "sequence" not in self.results.columns:
            raise ValueError("Master DataFrame must contain a 'sequence' column.")

    def evaluate(self) -> pd.DataFrame:
        mask = self.results["phase1_pass"] == True
        candidates = self.results.loc[mask, "sequence"]

        if candidates.empty:
            print("Phase 4: No sequences passed Phase 1. Skipping.")
            return self.results

        print("Running Phase 4: ToxinPred3 Toxicity Scorer...")

        original_cwd = os.getcwd()

        try:
            with tempfile.TemporaryDirectory() as tmpdir:
                os.chdir(tmpdir)

                fasta_path = os.path.join(tmpdir, "input.fasta")
                with open(fasta_path, "w") as f:
                    for idx, seq in candidates.items():
                        f.write(f">seq_{idx}\n{seq}\n")

                seq_list = list(candidates)

                tp.aac_comp(seq_list, "seq.aac")
                tp.dpc_comp(seq_list, "seq.dpc")

                os.system("sed -i 's/,$//g' seq.aac")
                os.system("sed -i 's/,$//g' seq.dpc")

                model_path = os.path.join(
                    os.path.dirname(tp.__file__),
                    "../model/toxinpred3.0_model.pkl"
                )

                original_loadtxt = np.loadtxt
                def safe_loadtxt(*args, **kwargs):
                    kwargs['ndmin'] = 2
                    return original_loadtxt(*args, **kwargs)

                np.loadtxt = safe_loadtxt

                try:
                    tp.prediction("seq.aac", "seq.dpc", model_path, "seq.pred")
                finally:
                    np.loadtxt = original_loadtxt

                tp.class_assignment("seq.pred", 0.38, "seq.out")

                if not os.path.exists("seq.out"):
                    print("  ToxinPred3 produced no output file.")
                    return self.results

                df_out = pd.read_csv("seq.out").fillna(0)

                candidate_indices = list(candidates.index)

                for i, row in df_out.iterrows():
                    if i >= len(candidate_indices):
                        break
                    original_idx = candidate_indices[i]
                    ml_score     = float(row.get("ML Score", 0))

                    phase4_score = round((1.0 - ml_score) * 100.0, 2)
                    self.results.at[original_idx, "phase4_score"] = phase4_score

        finally:
            os.chdir(original_cwd)

        scored     = (self.results.loc[mask, "phase4_score"] > 0).sum()
        n_toxic    = (self.results.loc[mask, "phase4_score"] < 62).sum()
        print(f"    Phase 4 Complete: {scored}/{mask.sum()} sequences scored.")
        print(f"    Predicted toxic (phase4_score < 62): {n_toxic}\n")

        return self.results

# Set up LLAMP

In [ ]:
import os, sys, math, difflib, importlib.util
import numpy as np
import torch
import pandas as pd
from transformers import AutoTokenizer
import gdown

DRIVE_PATH     = "/content/drive/MyDrive/DBAASP_Bioinformatics"
LLAMP_WEIGHTS  = f"{DRIVE_PATH}/LLAMP_weights/llamp_weights.pt"
ESM_DIR        = f"{DRIVE_PATH}/LLAMP_weights/peptide_tuned_ESM-2"
DEVICE         = torch.device("cuda" if torch.cuda.is_available() else "cpu")

os.makedirs(ESM_DIR, exist_ok=True)

if not os.path.exists("/content/LLAMP"):
    os.system("git clone https://github.com/GIST-CSBL/LLAMP.git /content/LLAMP")

for p in ["/content/LLAMP", "/content/LLAMP/utils"]:
    if p not in sys.path: sys.path.append(p)

if not os.path.exists(LLAMP_WEIGHTS):
    print("Downloading LLAMP weights...")
    gdown.download(id="1hmfL7uRZsHo4pn0o0nqaqPcntxGJIhE7",
                   output=LLAMP_WEIGHTS, quiet=False)

if not os.path.exists(f"{ESM_DIR}/model.safetensors"):
    print("Downloading ESM-2 weights...")
    from transformers import AutoModel
    AutoTokenizer.from_pretrained("Daehun/peptide_tuned_ESM-2").save_pretrained(ESM_DIR)
    AutoModel.from_pretrained("Daehun/peptide_tuned_ESM-2").save_pretrained(ESM_DIR)

spec = importlib.util.spec_from_file_location(
    "llamp_model", "/content/LLAMP/utils/model.py"
)
mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(mod)

llamp_model = mod.LLAMP(pretrained_model=ESM_DIR, pooling="mean")
llamp_model.load_state_dict(
    torch.load(LLAMP_WEIGHTS, map_location=DEVICE), strict=False
)
llamp_model.to(DEVICE).eval()
llamp_tokenizer = AutoTokenizer.from_pretrained(ESM_DIR)

genomic_data = torch.load(
    "/content/LLAMP/data/Genomic_featrues/genome_features.pt",
    map_location="cpu", weights_only=False
)
AVAILABLE_ORGANISMS = sorted(genomic_data.keys())
print(f"LLAMP ready on {DEVICE}. {len(AVAILABLE_ORGANISMS)} organisms available.")


def _get_genome_tensor(organism_name, target_dim=340):
    matches = difflib.get_close_matches(
        organism_name, AVAILABLE_ORGANISMS, n=1, cutoff=0.3
    )
    if not matches:
        raise ValueError(
            f"Organism '{organism_name}' not found. "
            f"Try one of: {AVAILABLE_ORGANISMS[:5]}"
        )
    matched = matches[0]
    raw     = genomic_data[matched]
    if isinstance(raw, list) and len(raw) == 1: raw = raw[0]
    if hasattr(raw, "ndim") and raw.ndim > 1:   raw = raw[0]
    if hasattr(raw, "tolist"): raw = raw.tolist()
    nums = []
    for item in (raw if isinstance(raw, (list, np.ndarray)) else [raw]):
        for sub in (item if isinstance(item, (list, np.ndarray)) else [item]):
            if isinstance(sub, (int, float, np.number)):
                nums.append(float(sub))
    t = torch.tensor(nums).float().unsqueeze(0)
    if   t.shape[-1] > target_dim: t = t[:, :target_dim]
    elif t.shape[-1] < target_dim: t = torch.nn.functional.pad(
        t, (0, target_dim - t.shape[-1])
    )
    return t.to(DEVICE), matched


def _predict_mic(sequence, genome_tensor):
    inputs = llamp_tokenizer(
        sequence, return_tensors="pt",
        padding=True, truncation=True, max_length=512
    )
    with torch.no_grad():
        log2_mic = llamp_model(
            inputs["input_ids"].to(DEVICE),
            inputs["attention_mask"].to(DEVICE),
            genome_tensor
        )
    mic = round(2 ** float(log2_mic.item()), 4)
    score = round(max(0.0, min(100.0, 100.0 * (1.0 - math.log2(max(mic, 0.01)) / 7.0))), 2)
    return mic, score

# Unified evaluation router


In [ ]:
import os, re
import pandas as pd


VALID_AAS = set("ACDEFGHIKLMNPQRSTVWY")

def _is_valid_sequence(s):
    s = s.strip().upper()
    return len(s) >= 4 and all(c in VALID_AAS for c in s)

def _load_sequences_from_file(path):
    """
    Reads a .fasta, .fa, .txt, or .csv file.
    Returns a list of (sequence, organism_or_None) tuples.
    organism is None unless the file has an 'organism' column.
    """
    ext = os.path.splitext(path)[1].lower()

    if ext in (".fasta", ".fa"):
        seqs = []
        with open(path) as f:
            buf = []
            for line in f:
                line = line.strip()
                if not line: continue
                if line.startswith(">"):
                    if buf: seqs.append(("".join(buf).upper(), None))
                    buf = []
                else:
                    buf.append(line)
            if buf: seqs.append(("".join(buf).upper(), None))
        return seqs

    if ext == ".txt":
        seqs = []
        with open(path) as f:
            for line in f:
                s = line.strip().upper()
                if s and _is_valid_sequence(s):
                    seqs.append((s, None))
        return seqs

    if ext == ".csv":
        df = pd.read_csv(path)
        df.columns = [c.strip().lower() for c in df.columns]
        if "sequence" not in df.columns:
            raise ValueError(f"CSV must have a 'sequence' column. Found: {list(df.columns)}")
        org_col = "organism" if "organism" in df.columns else None
        seqs = []
        for _, row in df.iterrows():
            seq = str(row["sequence"]).strip().upper()
            org = str(row[org_col]).strip() if org_col else None
            if _is_valid_sequence(seq):
                seqs.append((seq, org))
        return seqs

    raise ValueError(f"Unsupported file type: {ext}. Use .fasta, .txt, or .csv")


def _run_phases_1_to_4(sequences):
    df = pd.DataFrame({"sequence": sequences})

    p1 = Phase1_PhysicochemicalGate(df.copy())
    df = p1.evaluate()
    p2 = Phase2_NoveltyScorer(df, db_path=AMP_DB_PATH)
    df = p2.evaluate()
    p3 = Phase3_AMPlifyScorer(df)
    df = p3.evaluate()
    p4 = Phase4_ToxicityScorer(df)
    df = p4.evaluate()

    return df


def _run_llamp(sequence_organism_pairs):

    rows = []

    from collections import defaultdict
    by_org = defaultdict(list)
    for seq, org in sequence_organism_pairs:
        by_org[org].append(seq)

    for org, seqs in by_org.items():
        try:
            genome_tensor, matched = _get_genome_tensor(org)
            print(f"  LLAMP organism: '{matched}' ({len(seqs)} sequences)")
        except ValueError as e:
            print(f"  Skipping '{org}': {e}")
            for seq in seqs:
                rows.append({
                    "sequence": seq, "organism": org,
                    "llamp_mic_ug_ml": None, "llamp_score": None
                })
            continue

        for seq in seqs:
            try:
                mic, score = _predict_mic(seq, genome_tensor)
            except Exception as e:
                print(f"  Warning: failed on '{seq[:20]}': {e}")
                mic, score = None, None
            rows.append({
                "sequence":       seq,
                "organism":       matched,
                "llamp_mic_ug_ml": mic,
                "llamp_score":    score,
            })

    return pd.DataFrame(rows)


def evaluate(input_data, organism=None, save_dir=None):

    is_file   = False
    file_path = None

    if isinstance(input_data, str) and os.path.exists(input_data):
        is_file   = True
        file_path = input_data
        pairs     = _load_sequences_from_file(file_path)
    elif isinstance(input_data, str) and _is_valid_sequence(input_data):
        pairs = [(input_data.strip().upper(), organism)]
    elif isinstance(input_data, list):
        pairs = [(s.strip().upper(), organism) for s in input_data if _is_valid_sequence(s)]
    else:
        raise ValueError(
            "input_data must be a peptide sequence string, a list of sequences, "
            "or a path to a .fasta / .txt / .csv file."
        )

    if not pairs:
        print("No valid sequences found in input.")
        return pd.DataFrame()

    has_organisms = any(org is not None for _, org in pairs)
    mode = 2 if (has_organisms or organism is not None) else 1

    print(f"Input: {len(pairs)} sequence(s)  |  Mode: {'LLAMP (organism-specific)' if mode == 2 else 'Phases 1-4'}")

    if mode == 1:
        sequences = [seq for seq, _ in pairs]
        result_df = _run_phases_1_to_4(sequences)

        output_cols = [
            "sequence",
            "phase1_pass",
            # "phase1_charge",
            # "phase1_hydrophobic_frac",
            # "phase1_hydrophobic_moment",
            # "phase2_blast_identity",
            # "phase2_entropy",
            "phase2_score",
            "phase3_score",
            "phase4_score",
        ]
        output_cols = [c for c in output_cols if c in result_df.columns]
        result_df   = result_df[output_cols]

    else:
        if organism is not None and not has_organisms:
            pairs = [(seq, organism) for seq, _ in pairs]

        result_df = _run_llamp(pairs)
        result_df = result_df[["sequence", "organism", "llamp_mic_ug_ml", "llamp_score"]]
        result_df = result_df.sort_values("llamp_score", ascending=False, na_position="last")
        result_df = result_df.reset_index(drop=True)

    if is_file or save_dir is not None:
        if save_dir is None:
            save_dir = os.path.dirname(os.path.abspath(file_path)) if file_path else DRIVE_PATH
        os.makedirs(save_dir, exist_ok=True)

        base = os.path.splitext(os.path.basename(file_path))[0] if file_path else "input"
        suffix = "llamp" if mode == 2 else "evaluation"
        out_path = os.path.join(save_dir, f"{base}_{suffix}.csv")

        result_df.to_csv(out_path, index=False)
        print(f"\nSaved → {out_path}")

    print(f"\nResults ({len(result_df)} sequences):")
    if mode == 1:
        n_pass = result_df["phase1_pass"].sum() if "phase1_pass" in result_df.columns else "?"
        print(f"  Phase 1 pass: {n_pass}/{len(result_df)}")
        for col in ["phase2_score", "phase3_score", "phase4_score"]:
            if col in result_df.columns:
                vals = result_df[col].dropna()
                if len(vals):
                    print(f"  {col}: mean={vals.mean():.1f}  min={vals.min():.1f}  max={vals.max():.1f}")
    else:
        vals = result_df["llamp_score"].dropna()
        if len(vals):
            print(f"  LLAMP score: mean={vals.mean():.1f}  min={vals.min():.1f}  max={vals.max():.1f}")
            print(f"  Predicted MIC range: {result_df['llamp_mic_ug_ml'].min():.2f} – {result_df['llamp_mic_ug_ml'].max():.2f} µg/mL")
        print(f"\n  Top 5:")
        print(result_df.head(5).to_string(index=False))

    return result_df

In [ ]:
from google.colab import drive

drive.flush_and_unmount()

drive.mount('/content/drive', force_remount=True)

In [ ]:
import torch

absolute_data_path = "/content/LLAMP/data/Genomic_featrues/genome_features.pt"

genomic_data = torch.load(absolute_data_path, map_location="cpu", weights_only=False)

if isinstance(genomic_data, dict):
    available_species = list(genomic_data.keys())

    print(f"--- Total Organisms Found: {len(available_species)} ---\n")
    for idx, species in enumerate(sorted(available_species), 1):
        print(f"{idx}. {species}")
else:
    print("The file structure is not a dictionary. Cannot extract organism names directly.")

In [ ]:
import torch

absolute_data_path = "/content/LLAMP/data/Genomic_featrues/genome_features.pt"
genomic_data = torch.load(absolute_data_path, map_location="cpu", weights_only=False)

if isinstance(genomic_data, dict):
    available_species = list(genomic_data.keys())

    print(f"--- Total Organisms Found: {len(available_species)} ---\n")
    for idx, species in enumerate(sorted(available_species), 1):
        print(f"{idx}. {species}")
else:
    print("The file structure is not a dictionary. Cannot extract organism names directly.")

In [ ]:
result = evaluate("/content/drive/MyDrive/DBAASP_Bioinformatics/56pairs.csv")

In [ ]:
import sklearn
print(sklearn.__version__)

# Phases 1-4

## Single Sequence

In [ ]:
result = evaluate("KLLLKLLKKLLLKLLK")

## List of sequences

In [ ]:
result = evaluate(["KLLLKLLK", "RWWRWWRW", "GIGAVLKVLTTGLPALI"])

## File of sequences

In [ ]:
result = evaluate("/content/drive/MyDrive/DBAASP_Bioinformatics/generated/escherichia_coli.txt")

In [ ]:
result = evaluate("/content/drive/MyDrive/DBAASP_Bioinformatics/peptide_sequences_only.csv")

In [ ]:
result = evaluate("/content/drive/MyDrive/DBAASP_Bioinformatics/escherichia_coli_rec.fasta")

# LLAMP

## Single pair

In [ ]:
result = evaluate("KLLLKLLKKLLLKLLK", organism="Escherichia coli")

## File of pairs (Sequence-Organism)

In [ ]:
result = evaluate("/content/drive/MyDrive/DBAASP_Bioinformatics/peptide_organism_pairs_LLAMP.csv")